In [0]:
# Create environment widget
dbutils.widgets.dropdown("environment", "prod", ["dev", "preprod", "prod"], "Environment")

In [0]:
%run /Workspace/Users/lakshmisas96@gmail.com/orders-project/orders-analytics-pipeline/utils/config_loader

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col

config = load_config_from_widget()

# Logging
print(f"🚀 Bronze Orders Pipeline - {config['environment'].upper()} Environment")
print("Status: Running bronze ingestion...")

# Read source data
orders_path = config['storage']['landing']['orders']
df_orders = (spark.read.format("parquet").load(orders_path)
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("environment", lit(config['environment']))
)

# External table location
external_path = f"{config['storage']['external']['bronze']}/orders_raw"
print(f"📂 Writing to: {external_path}")

# Write to external location (creates _delta_log automatically!)
df_orders.write.format("delta").mode("overwrite").save(external_path)

# Create external table pointing to this location
bronze_schema = config['schemas']['bronze']
bronze_table = f"{config['catalog']}.{bronze_schema}.orders_raw"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {config['catalog']}.{bronze_schema}")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {bronze_table}
    USING DELTA
    LOCATION '{external_path}'
""")

print(f"✅ {bronze_table}: {spark.table(bronze_table).count()} records")
print(f"✅ Delta logs created at: {external_path}/_delta_log/")